In [1]:
import json
from langchain_core.documents import Document

## 1. Creating LangChain Documents

In [2]:
def load_rules(path):
    """Load the rule cards created in notebook 01."""
    with open(path) as f:
        return json.load(f)

def rules_to_documents(rules):
    """Turn each rule card into a LangChain Document."""
    output = []
    for rule in rules:
        output.append(
            
            Document(
                page_content=rule["text"],
                metadata={
                    "rule_id": rule["rule_id"],
                    "title": rule["title"],
                    "updated": rule["updated"],
                    "chapter": rule["chapter"],
                    "start_page": rule["start_page"],
                    "end_page": rule["end_page"],
                    "printed_start": rule["printed_start"],
                    "printed_end": rule["printed_end"],
                }
            )

        )

    return output

rules = load_rules("../data/processed/b3_rules.json")
documents = rules_to_documents(rules)

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:200])

123
{'rule_id': 'B3-1-01', 'title': 'Comprehensive Risk Assessment', 'updated': '11/05/2025', 'chapter': 'B3-1', 'start_page': 286, 'end_page': 288, 'printed_start': 268, 'printed_end': 269}
B3-1-01, Comprehensive Risk Assessment (11/05/2025)
Introduction
This topic contains information on the comprehensive risk assessment approach to underwriting, including:
Overview
Comprehensive Risk A


## 2. Chunking

In [3]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1500, chunk_overlap=200):
    """Split long documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks = split_documents(documents)

print("Total chunks:", len(chunks))

per_rule = Counter(c.metadata["rule_id"] for c in chunks)
print("Rules kept whole:", sum(1 for n in per_rule.values() if n == 1))
print("Most-split rules:", per_rule.most_common(3))

Total chunks: 401
Rules kept whole: 34
Most-split rules: [('B3-5.3-09', 18), ('B3-2-02', 15), ('B3-6-05', 13)]


In [4]:
dti_chunks = [c for c in chunks if c.metadata["rule_id"] == "B3-6-02"]
for i, c in enumerate(dti_chunks):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150])


--- Chunk 0 (1478 chars) ---
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI 

--- Chunk 1 (1482 chars) ---
including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refina

--- Chunk 2 (1488 chars) ---
subordinate lien payments) or rental payments (see B3-6-05, Monthly Debt Obligations);
if the subject loan is a second home or investment property, us

--- Chunk 3 (1416 chars) ---
when qualifying borrowers. This is acceptable as long as Fannie Mae’s minimum requirements are met,
and lenders consistently apply the same approach t

--- Chunk 4 (1451 chars) ---
In all cases, if the lender determines that there is new subordinate financing on the subject property during the
loan process, the mortgage loan must

--- Chunk 5 (1462 chars) ---
recalculated outside of DU.
4
• If the recalculated DTI ratio exc

In [5]:
def add_chunk_headers(chunks):
    """Return new chunks with a context header and a chunk_index"""
    position = Counter()
    output = []
    for chunk in chunks:
        rule_id = chunk.metadata["rule_id"]
        header = f"Fannie Mae Selling Guide, {rule_id}: {chunk.metadata["title"]}\n\n"
        output.append(
            Document(
                page_content=header + chunk.page_content,
                metadata={**chunk.metadata, "chunk_index": position[rule_id]},
            )
        )
        position[rule_id] += 1
    return output

chunks_with_headers = add_chunk_headers(chunks)

dti = [c for c in chunks_with_headers if c.metadata["rule_id"] == "B3-6-02"]
print("\nChunk 1 metadata:", dti[1].metadata)
print("\nChunk 1 text:\n" + dti[1].page_content[:250])


Chunk 1 metadata: {'rule_id': 'B3-6-02', 'title': 'Debt-to-Income Ratios', 'updated': '04/02/2025', 'chapter': 'B3-6', 'start_page': 524, 'end_page': 528, 'printed_start': 506, 'printed_end': 509, 'chunk_index': 1}

Chunk 1 text:
Fannie Mae Selling Guide, B3-6-02: Debt-to-Income Ratios

including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refinance Transactions);
high LTV refinance tran


## 3. Embedding Model

In [6]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

# def first_chunk(rule_id):
#     """Return the first chunk of a given rule."""
#     return next(
#         c for c in chunks_with_headers
#         if c.metadata["rule_id"] == rule_id and c.metadata["chunk_index"] == 0
#     )

# def compare(question, samples):
#     """Print how similar a question is to each sample chunk."""
#     question_vector = embedding_model.embed_query(question)
#     chunk_vectors = embedding_model.embed_documents([c.page_content for c in samples])
#     print(f"\nQuestion: {question}")
#     for chunk, vector in zip(samples, chunk_vectors):
#         similarity = np.dot(question_vector, vector)
#         print(f"  {similarity:.3f}  {chunk.metadata['rule_id']}: {chunk.metadata['title']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
# samples = [
#     first_chunk("B3-6-02"),     # Debt-to-Income Ratios
#     first_chunk("B3-5.1-01"),   # Credit Scores
#     first_chunk("B3-4.3-04"),   # Personal Gifts
# ]

# # What does an embedding look like?
# vector = embedding_model.embed_query("test")
# print("Embedding length:", len(vector))
# print("First 5 numbers:", np.round(vector[:5], 3))

# # Compare two different questions against the same three chunks
# compare("How much debt can a borrower have compared to their income?", samples)
# compare("Can my parents help me with the down payment?", samples)

## 4. Vector Database

In [7]:
from langchain_qdrant import QdrantVectorStore

COLLECTION = "selling_guide_b3"

def build_vector_store(chunks, embedding_model, path, collection_name):
    """Embed chunks and store them in a local Qdrant collection"""
    return QdrantVectorStore.from_documents(
        documents=chunks, 
        embedding=embedding_model,
        path=path,
        collection_name=collection_name,
        force_recreate=True
    )

vector_store = build_vector_store(
    chunks_with_headers, embedding_model, "../data/qdrant", COLLECTION
)


print("Points stored:", vector_store.client.count(COLLECTION).count)
print("Vector settings:", vector_store.client.get_collection(COLLECTION).config.params.vectors)

Points stored: 401
Vector settings: {'': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}


In [8]:
def search(question, k=3):
    """Find and print the top-k matching chunks for a question"""
    results = vector_store.similarity_search_with_score(question, k=k)
    
    print(f"\nQuestion: {question}")
    for doc, score in results:
        m = doc.metadata
        print(f" {score:.3f} {m['rule_id']} (chunk {m['chunk_index']}): "
              f"{m['title']}, p. {m['printed_start']}")

search("What is the maximum debt-to-income ratio for a manually underwritten loan?")
search("How long must a borrower wait after a bankruptcy?")
search("Can my parents help me with the down payment?")


Question: What is the maximum debt-to-income ratio for a manually underwritten loan?
 0.823 B3-6-02 (chunk 1): Debt-to-Income Ratios, p. 506
 0.795 B3-6-02 (chunk 0): Debt-to-Income Ratios, p. 506
 0.790 B3-6-02 (chunk 4): Debt-to-Income Ratios, p. 506

Question: How long must a borrower wait after a bankruptcy?
 0.826 B3-5.3-07 (chunk 3): Significant Derogatory Credit Events — Waiting Periods and Re-establishing Credit, p. 481
 0.809 B3-5.3-07 (chunk 4): Significant Derogatory Credit Events — Waiting Periods and Re-establishing Credit, p. 481
 0.761 B3-5.3-07 (chunk 7): Significant Derogatory Credit Events — Waiting Periods and Re-establishing Credit, p. 481

Question: Can my parents help me with the down payment?
 0.682 B3-4.3-06 (chunk 5): Grants and Lender Contributions, p. 438
 0.660 B3-4.3-08 (chunk 2): Employer Assistance, p. 442
 0.649 B3-4.2-03 (chunk 1): Individual Development Accounts, p. 427


In [9]:
eval_set = [
    {"question": "What is the maximum debt-to-income ratio for a manually underwritten loan?", "expected": ["B3-6-02"]},
    {"question": "How long must a borrower wait after a bankruptcy?", "expected": ["B3-5.3-07"]},
    {"question": "Can my parents help me with the down payment?", "expected": ["B3-4.3-04"]},
    {"question": "What is the minimum credit score for a fixed-rate loan?", "expected": ["B3-5.1-01"]},
    {"question": "What counts as a large deposit in a bank statement?", "expected": ["B3-4.2-02"]},
    {"question": "When is a borrower considered self-employed?", "expected": ["B3-3.5-01"]},
    {"question": "Can I use cryptocurrency for my down payment?", "expected": ["B3-4.1-04"]},
    {"question": "How do I calculate monthly income if I'm paid every two weeks?", "expected": ["B3-3.3-01"]},
    {"question": "How many months of reserves are needed for an investment property?", "expected": ["B3-4.1-01"]},
    {"question": "What payment is used for a deferred student loan?", "expected": ["B3-6-05"]},
    {"question": "Can rental income from the home I'm moving out of help me qualify?", "expected": ["B3-3.8-05"]},
    {"question": "When must the lender verify the borrower is still employed before closing?", "expected": ["B3-3.1-04"]},
    {"question": "How long after a foreclosure can someone get a new mortgage?", "expected": ["B3-5.3-07"]},
    {"question": "Can a future pay raise be counted as income?", "expected": ["B3-3.2-02"]},
    {"question": "Can Social Security income be grossed up?", "expected": ["B3-3.4-15"]},
    {"question": "How much can the seller contribute toward closing costs?", "expected": ["B3-4.1-02"]},
    {"question": "What monthly payment is used for a credit card if none is shown on the credit report?", "expected": ["B3-6-05"]},
    {"question": "Which score is used when a borrower has three credit scores?", "expected": ["B3-5.1-02"]},
    {"question": "What documents are needed to use child support as income?", "expected": ["B3-3.4-02"]},
    {"question": "Can the seller give the buyer a gift of equity?", "expected": ["B3-4.3-05"]},
]


def first_correct_rank(question, expected, k=10):
    """Return the rank (1 = top) of the first result from an expected rule, or None."""
    results = vector_store.similarity_search(question, k=k)
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["rule_id"] in expected:
            return rank
    return None


def evaluate_retrieval(eval_set, ks=(1, 3, 5)):
    """Print Hit@k scores and the questions that missed the top 5."""
    ranks = [first_correct_rank(item["question"], item["expected"]) for item in eval_set]

    for k in ks:
        hits = sum(1 for r in ranks if r is not None and r <= k)
        print(f"Hit@{k}: {hits}/{len(eval_set)} = {hits / len(eval_set):.0%}")

    print("\nMissed the top 5:")
    for item, rank in zip(eval_set, ranks):
        if rank is None or rank > 5:
            where = "not in top 10" if rank is None else f"rank {rank}"
            print(f"  [{where}] {item['question']}  (expected {item['expected']})")

    return ranks


ranks = evaluate_retrieval(eval_set)

Hit@1: 17/20 = 85%
Hit@3: 19/20 = 95%
Hit@5: 20/20 = 100%

Missed the top 5:


## 5. LLM

In [10]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()    # reads .env so the API key is available

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

response = llm.invoke("In one sentence, what is a debt-to-income ratio?")

print(response.content)
print(response.usage_metadata)

A debt‑to‑income ratio is the percentage of a person’s gross monthly income that goes toward paying all of their recurring debt obligations.
{'input_tokens': 83, 'output_tokens': 67, 'total_tokens': 150, 'output_token_details': {'reasoning': 30}}


In [11]:
from langchain_core.prompts import ChatPromptTemplate

# Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def format_context(docs):
    """Format chunks as numbered sources with rule ID, title, and pages."""
    blocks = []
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        label = (
                 f"[Source {i}] {m['rule_id']}: {m['title']} "
                 f"(pages {m['printed_start']}-{m['printed_end']})"
        )
        blocks.append(f"{label}\n{doc.page_content}")
    return "\n\n".join(blocks)
        


SYSTEM_PROMPT = """You are an assistant that answers questions about the Fannie Mae Selling Guide, Part B3 (Underwriting Borrowers).

Rules:
- Answer ONLY using the sources provided. Do not use outside knowledge.
- Cite every fact with its rule ID and pages, like (B3-6-02, pp. 506-509).
- If the sources do not contain the answer, say: "The provided sections do not answer this question."
- Be concise and precise. Include exact numbers and conditions when the sources give them."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Question: {question}\n\nSources:\n{context}"),
])

def ask(question):
    """Retrive sources, ask the LLM, and print the answer with its sources."""
    docs = retriever.invoke(question)
    messages = prompt.invoke({"question": question, "context": format_context(docs)})
    response = llm.invoke(messages)

    print(f"Question: {question}\n")
    print(response.content)
    print("\nRetrieved:", ", ".join(d.metadata["rule_id"] for d in docs))
    print("-" * 80)

# Test
ask("What is the maximum debt-to-income ratio for a manually underwritten loan?")
ask("Can my parents help me with the down payment?")
ask("What are the appraisal requirements for a condo?")

Question: What is the maximum debt-to-income ratio for a manually underwritten loan?

The guide sets the baseline maximum total DTI for a manually‑underwritten loan at **36 % of the borrower’s stable monthly income**. If the borrower satisfies the credit‑score and reserve requirements in the Eligibility Matrix, the DTI may be increased **up to 45 %** (but not beyond) (B3-6-02, pp. 506‑509).

Retrieved: B3-6-02, B3-6-02, B3-6-02, B3-6-02, B3-6-02
--------------------------------------------------------------------------------
Question: Can my parents help me with the down payment?

Yes. Your parents can provide a **personal gift** that can be used toward the down‑payment (and, after the borrower’s minimum contribution is met, also toward closing costs and reserves).

**What the guide requires**

* A gift from an “acceptable donor” – which includes a parent – must be documented with a **gift letter** that:
  * States the actual or maximum dollar amount of the gift;
  * Includes the donor

In [13]:
rules_by_id = {rule["rule_id"]: rule for rule in rules}

chunk_retriever = vector_store.as_retriever(search_kwargs={"k": 8})

# Search small, read big
def retrieve_rules(question, max_rules=3):
    """Find relevant chunks, then return the full rules they belong to."""
    docs = chunk_retriever.invoke(question)
    rule_ids = []
    for doc in docs:
        rule_id = doc.metadata["rule_id"]
        if rule_id not in rule_ids:
            rule_ids.append(rule_id)
    return [rules_by_id[rule_id] for rule_id in rule_ids[:max_rules]]

def format_rules(rule_list):
    """Format full rules as numbered sources."""
    blocks = []
    for i, rule in enumerate(rule_list, start=1):
        label = (f"[Source {i}] {rule['rule_id']}: {rule['title']} "
                 f"(pages {rule['printed_start']}-{rule['printed_end']})")
        blocks.append(f"{label}\n{rule['text']}")
    return "\n\n".join(blocks)

def ask_rules(question, max_rules=3):
    """RAG with full rules as context."""
    rule_list = retrieve_rules(question, max_rules)
    messages = prompt.invoke({"question": question, "context": format_rules(rule_list)})
    response = llm.invoke(messages)

    print(f"Question: {question}\n")
    print(response.content)
    print("\nRules used:", ", ".join(r["rule_id"] for r in rule_list))
    print("Tokens:", response.usage_metadata)
    print("-" * 80)

ask_rules("Can my parents help me with the down payment?")
ask_rules("What is the maximum debt-to-income ratio for a manually underwritten loan?")

Question: Can my parents help me with the down payment?

Parents are not listed among the entities that may provide a grant‑type contribution for the down payment. The Fannie Mae guide states that grant funds must come from a federal, state, local, housing‑finance, nonprofit, Federal Home Loan Bank, employer, Indian tribe, or the lender itself (B3‑4.3‑06, pp. 438‑441). Since parents are not one of those allowable sources, they cannot provide a grant‑type down‑payment assistance under the rules cited.

Rules used: B3-4.3-06, B3-4.3-08, B3-4.2-03
Tokens: {'input_tokens': 3179, 'output_tokens': 434, 'total_tokens': 3613, 'output_token_details': {'reasoning': 319}}
--------------------------------------------------------------------------------
Question: What is the maximum debt-to-income ratio for a manually underwritten loan?

For a manually‑underwritten loan the standard ceiling is **36 % of the borrower’s stable monthly income**. If the borrower satisfies the credit‑score and reserve t

In [14]:
q = "Can my parents help me with the down payment?"

BGE_INSTRUCTION = "Represent this sentence for searching relevant passages: "

query_embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},           # used for documents
    query_encode_kwargs={"normalize_embeddings": True,      # used for questions
                         "prompt": BGE_INSTRUCTION},
)
v_plain = embedding_model.embed_query(q)          # without instruction
v_instr = query_embedding_model.embed_query(q)    # with instruction

print("Without:", np.round(v_plain[:5], 3))
print("With:   ", np.round(v_instr[:5], 3))
print("Similarity between the two:", round(float(np.dot(v_plain, v_instr)), 3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Without: [ 0.008 -0.032  0.026 -0.062 -0.021]
With:    [ 0.019 -0.021  0.021 -0.059 -0.02 ]
Similarity between the two: 0.975


In [15]:
# 1. Same model, but questions get BGE's search instruction
BGE_INSTRUCTION = "Represent this sentence for searching relevant passages: "

query_embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},           # used for documents
    query_encode_kwargs={"normalize_embeddings": True,      # used for questions
                         "prompt": BGE_INSTRUCTION},
)

# 2. Reuse the existing collection (stored chunks stay exactly as they are)
vector_store_instr = QdrantVectorStore(
    client=vector_store.client,
    collection_name=COLLECTION,
    embedding=query_embedding_model,
)


# 3. Evaluation that works with any vector store
def first_correct_rank(store, question, expected, k=10):
    """Rank (1 = top) of the first result from an expected rule, or None."""
    results = store.similarity_search(question, k=k)
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["rule_id"] in expected:
            return rank
    return None


def evaluate_retrieval(store, eval_set, ks=(1, 3, 5)):
    """Print Hit@k and the questions that weren't ranked first."""
    ranks = [first_correct_rank(store, item["question"], item["expected"]) for item in eval_set]
    for k in ks:
        hits = sum(1 for r in ranks if r is not None and r <= k)
        print(f"  Hit@{k}: {hits}/{len(eval_set)} = {hits / len(eval_set):.0%}")
    print("  Not ranked first:")
    for item, rank in zip(eval_set, ranks):
        if rank != 1:
            print(f"    [rank {rank}] {item['question']}")
    return ranks


# 4. Compare
print("WITHOUT instruction:")
ranks_before = evaluate_retrieval(vector_store, eval_set)

print("\nWITH instruction:")
ranks_after = evaluate_retrieval(vector_store_instr, eval_set)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

WITHOUT instruction:
  Hit@1: 17/20 = 85%
  Hit@3: 19/20 = 95%
  Hit@5: 20/20 = 100%
  Not ranked first:
    [rank 5] Can my parents help me with the down payment?
    [rank 2] How do I calculate monthly income if I'm paid every two weeks?
    [rank 2] How many months of reserves are needed for an investment property?

WITH instruction:
  Hit@1: 17/20 = 85%
  Hit@3: 19/20 = 95%
  Hit@5: 20/20 = 100%
  Not ranked first:
    [rank 5] Can my parents help me with the down payment?
    [rank 2] How do I calculate monthly income if I'm paid every two weeks?
    [rank 2] How many months of reserves are needed for an investment property?


In [16]:
import time
from langchain_core.prompts import ChatPromptTemplate

# 1. General rewriting instructions (no hints from the test set)
REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You rewrite questions about mortgage lending so they match the formal wording "
     "used in mortgage underwriting guidelines. Keep the original meaning. Replace casual "
     "or everyday phrasing with the standard industry terms. "
     "Return only the rewritten question, nothing else."),
    ("human", "{question}"),
])


def rewrite_query(question):
    """Ask the LLM to rephrase a question in formal underwriting language."""
    messages = REWRITE_PROMPT.invoke({"question": question})
    return llm.invoke(messages).content.strip()


# 2. Rewrite all test questions (pause to stay within rate limits)
rewrites = []
for item in eval_set:
    rewrites.append(rewrite_query(item["question"]))
    time.sleep(2)

# 3. Inspect
for item, rewritten in zip(eval_set, rewrites):
    print(f"• {item['question']}\n  → {rewritten}\n")

# 4. Evaluate retrieval with the rewritten questions
rewritten_set = [
    {"question": rewritten, "expected": item["expected"]}
    for item, rewritten in zip(eval_set, rewrites)
]

print("BASELINE (original questions):")
evaluate_retrieval(vector_store, eval_set)

print("\nWITH QUERY REWRITING:")
evaluate_retrieval(vector_store, rewritten_set)

• What is the maximum debt-to-income ratio for a manually underwritten loan?
  → What is the maximum permissible debt‑to‑income (DTI) ratio for a manually underwritten mortgage loan?

• How long must a borrower wait after a bankruptcy?
  → What is the minimum post‑bankruptcy waiting period before a borrower may be eligible for a new mortgage loan?

• Can my parents help me with the down payment?
  → Are gift funds from the borrower’s parents permissible for the down payment?

• What is the minimum credit score for a fixed-rate loan?
  → What is the minimum borrower credit score required for eligibility for a fixed‑rate mortgage loan?

• What counts as a large deposit in a bank statement?
  → What constitutes a significant deposit on a borrower's bank statement for underwriting purposes?

• When is a borrower considered self-employed?
  → Under what criteria is a borrower classified as self‑employed for underwriting purposes?

• Can I use cryptocurrency for my down payment?
  → Is the u

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 1, 1, 1, 1, 1, 8, 1, 1]

In [17]:
def rrf_merge(result_lists, k_rrf=60):
    """Merge several ranked lists of chunks using Reciprocal Rank Fusion."""
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            key = (doc.metadata["rule_id"], doc.metadata["chunk_index"])  # unique chunk ID
            scores[key] = scores.get(key, 0) + 1 / (k_rrf + rank)
            docs[key] = doc
    ordered = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ordered]


def first_correct_rank_in(docs, expected):
    """Rank (1 = top) of the first chunk from an expected rule, or None."""
    for rank, doc in enumerate(docs, start=1):
        if doc.metadata["rule_id"] in expected:
            return rank
    return None


# Search with both versions of each question, then merge
fused_ranks = []
for item, rewritten in zip(eval_set, rewrites):
    original_results = vector_store.similarity_search(item["question"], k=10)
    rewritten_results = vector_store.similarity_search(rewritten, k=10)
    merged = rrf_merge([original_results, rewritten_results])
    fused_ranks.append(first_correct_rank_in(merged, item["expected"]))

# Report
print("WITH FUSION (original + rewrite):")
for k in (1, 3, 5):
    hits = sum(1 for r in fused_ranks if r is not None and r <= k)
    print(f"  Hit@{k}: {hits}/{len(eval_set)} = {hits / len(eval_set):.0%}")
print("  Not ranked first:")
for item, rank in zip(eval_set, fused_ranks):
    if rank != 1:
        print(f"    [rank {rank}] {item['question']}")

WITH FUSION (original + rewrite):
  Hit@1: 16/20 = 80%
  Hit@3: 19/20 = 95%
  Hit@5: 20/20 = 100%
  Not ranked first:
    [rank 2] How do I calculate monthly income if I'm paid every two weeks?
    [rank 2] How many months of reserves are needed for an investment property?
    [rank 3] When must the lender verify the borrower is still employed before closing?
    [rank 4] Which score is used when a borrower has three credit scores?
